# Torsion Propagation & Internal - Cartesian Conversion -- Student NotebookChange a single backbone $$\psi_2$$ angle by $$+30^\circ$$, or flip a side-chain $$\chi_1$$ from $$-60^\circ$$ to $$+60^\circ$$. You touched **one** degree of freedom, yet every atom from the rotation point onward moves -- often by several angstroms. Why?This notebook lets you **play with** the answer:1. build a peptide backbone from **internal coordinates** (Z-matrix + NeRF algorithm);2. change ONE torsion and see the **downstream slice** rotate as a rigid block;3. extract the Z-matrix from Cartesian coordinates and verify the round-trip;4. explore the **IUPAC \(\leftrightarrow\) NeRF sign convention**.Companion reading: `TorsionPropagation_tutorial.md` (or the Chinese version).**Prerequisites:** `numpy`, `matplotlib`. (No RDKit needed.)Run the cells top to bottom, then try the exercises at the end.

## 0. SetupRun this notebook from the `internal2cartesian/` directory so that the `core` package imports.

In [ ]:
import os, sys# Make sure the directory containing `core/` is importable.if not os.path.isdir('core'):    if os.path.isdir('internal2cartesian/core'):        os.chdir('internal2cartesian')    else:        raise RuntimeError('Run this notebook from the internal2cartesian/ directory.')sys.path.insert(0, os.path.abspath('.'))import numpy as npimport matplotlib.pyplot as pltfrom mpl_toolkits.mplot3d import Axes3Dfrom core import (    build_peptide_from_internal,    extract_backbone_internal,    internal_to_cartesian,    cartesian_to_internal,    normalize,    place_atom,    peptide_ideal_geometry,    dihedral,    bond_angle,    ZMatrixEntry,    InternalCoords,)print('imports OK, cwd =', os.getcwd())

## 1. Ideal peptide geometry and the NeRF algorithmA peptide backbone is a serial kinematic chain. Bond **lengths** and bond **angles** are nearly constant (Engh & Huber ideal values); only the torsions $$\phi, \psi$$ vary per residue. This is the same assumption the `kinematics_loop` module uses.Let's look at the ideal geometry constants and the core forward-kinematics operation: `place_atom` (the NeRF algorithm).

In [ ]:
geo = peptide_ideal_geometry()print("Ideal peptide backbone geometry:")for k, v in geo.items():    if 'C_N' in k or 'N_CA' in k or 'CA_C' in k:        print(f"  {k:12s} = {v:.3f} A")    else:        print(f"  {k:12s} = {np.rad2deg(v):.1f} deg")print("\nThe NeRF construction: place_atom(a, b, c, bond, angle, torsion)")print("  Given three placed atoms A-B-C, places D such that:")print("    |C-D| = bond,  angle(B,C,D) = angle,  dihedral(A,B,C,D) = torsion")print()print("The local frame M = [bc_hat, n_cross_bc, n] maps the local offset into world space.")print("Local d = (-bond*cos(angle), bond*sin(angle)*cos(torsion), bond*sin(angle)*sin(torsion))")print("World D = C + M @ d_local")

## 2. Build a peptide backbone from internal coordinates`build_peptide_from_internal` walks down the backbone, placing each atom from its three predecessors using `place_atom`. Atom order:```N1, CA1, C1, N2, CA2, C2, N3, CA3, C3, N4, CA4, C4, N5, CA5, C5```The first three atoms are placed by hand; atoms 4+ use NeRF with $$\psi_{i-1}$$ (for each $$N_i$$), $$\omega=180^\circ$$ (for each $$CA_i$$), and $$\phi_i$$ (for each $$C_i$$).

In [ ]:
seq = "AAAAA"  # penta-alanineL = len(seq)# Build an alpha-helix: phi = -57 deg, psi = -47 degphi_helix = [-57.0] * Lpsi_helix = [-47.0] * Lcoords_ref, names = build_peptide_from_internal(seq, phi_deg=phi_helix, psi_deg=psi_helix)print(f'Sequence: {seq}  ({len(names)} backbone atoms)')print(f'End-to-end N1..C{L} span: {np.linalg.norm(coords_ref[-1] - coords_ref[0]):.2f} A')print(f'\nAtom coordinates:')for i, name in enumerate(names):    print(f'  [{i:2d}] {name:3s}  ({coords_ref[i, 0]:7.3f}, {coords_ref[i, 1]:7.3f}, {coords_ref[i, 2]:7.3f})')

In [ ]:
# 3D plot of the backbone.fig = plt.figure(figsize=(7, 5))ax = fig.add_subplot(projection='3d')ax.plot(coords_ref[:, 0], coords_ref[:, 1], coords_ref[:, 2], 'o-', lw=2, ms=5, color='tab:blue')for i, name in enumerate(names):    ax.text(coords_ref[i, 0], coords_ref[i, 1], coords_ref[i, 2], name, fontsize=7)ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')ax.set_title(f'Alpha-helical {seq}  (phi=-57, psi=-47)')plt.tight_layout(); plt.show()

## 3. The Rodrigues rotation operationRodrigues' formula rigidly rotates points about an axis:$$\mathbf{r}' = \mathbf{o} + (\mathbf{r} - \mathbf{o})\cos\theta + [\hat{\mathbf{k}} \times (\mathbf{r} - \mathbf{o})]\sin\theta + \hat{\mathbf{k}}[\hat{\mathbf{k}} \cdot (\mathbf{r} - \mathbf{o})](1 - \cos\theta)$$This is $$O(1)$$ per atom -- used by `kinematics_loop` for CCD and by RDKit's `SetDihedralDeg`.

In [ ]:
def rotate_points(points, origin, axis, theta):    # Rodrigues rotation: rotate points by theta (rad) about the line    # through origin with direction axis.    k = normalize(axis)    v = points - origin    cos_t, sin_t = np.cos(theta), np.sin(theta)    rotated = (v * cos_t               + np.cross(k, v) * sin_t               + np.outer(v @ k, k) * (1.0 - cos_t))    return origin + rotateddef apply_torsion_change(coords, a, b, c, slice_start, delta_deg):    # Rotate all atoms from slice_start onward about the B->C bond axis.    # coords: (n,3) array. a,b,c: 0-based. Axis is B->C, origin is C.    # slice_start: first atom index (0-based) to rotate.    # delta_deg: rotation angle in degrees.    origin = coords[c]    axis = coords[c] - coords[b]    delta_rad = np.deg2rad(delta_deg)    moved = coords.copy()    moved[slice_start:] = rotate_points(        coords[slice_start:], origin, axis, delta_rad    )    return movedprint('Rodrigues rotation and apply_torsion_change defined.')

## 4. Example 1: Rotate $$\psi_2$$ by $$+30^\circ$$$$\psi_2 = N_2 - CA_2 - C_2 - N_3$$ involves atoms 3,4,5,6 (0-based).- **Rotation axis**: $$CA_2 \rightarrow C_2$$ (atoms 4->5)- **Origin**: atom $$C_2$$ (index 5)- **Downstream slice**: starts at $$N_3$$ (index 6)Atoms 0-5 should be stationary; atoms 6-14 should all move.

In [ ]:
psi2_atoms = (3, 4, 5, 6)   # N2, CA2, C2, N3  (0-based)psi2_slice = 6               # N3 is the first atom that movescoords_psi = apply_torsion_change(    coords_ref, *psi2_atoms[:3], psi2_slice, delta_deg=+30.0)# Display displacement of each atom.disp = np.linalg.norm(coords_psi - coords_ref, axis=1)print(f'Rotating psi_2 by +30 deg\n')print(f'{"atom":>6s}  {"index":>5s}  {"displacement (A)":>18s}  {"moved?":>8s}')print('  ' + '-' * 42)for i, (name, d) in enumerate(zip(names, disp)):    marker = '  <<<' if i >= psi2_slice else ''    print(f'  {name:>6s}  {i:>5d}  {d:18.4f}{marker}')print(f'\nOK: Atoms 0-5 (N1...C2) are stationary.')print(f'OK: Atoms 6-14 (N3...C5) all moved -- by up to {disp[psi2_slice:].max():.2f} A.')print(f'  Max displacement at terminal atom C5: {disp[-1]:.2f} A')

In [ ]:
# Visualise displacement vs atom index.fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))# Bar chart of displacements.colors = ['0.5' if i < psi2_slice else 'tab:red' for i in range(len(names))]ax1.bar(range(len(names)), disp, color=colors, edgecolor='white')ax1.axvline(psi2_slice - 0.5, color='black', ls='--', lw=1.5, label='downstream slice start')ax1.set_xlabel('atom index'); ax1.set_ylabel('displacement (A)')ax1.set_title('Displacement after psi_2 +30 deg')ax1.legend(fontsize=8)ax1.set_xticks(range(len(names)))ax1.set_xticklabels(names, rotation=45, fontsize=6)# 3D comparison.ax2.remove()ax2 = fig.add_subplot(1, 2, 2, projection='3d')ax2.plot(coords_ref[:, 0], coords_ref[:, 1], coords_ref[:, 2],         'o-', color='0.5', lw=2, ms=4, label='reference (alpha-helix)')ax2.plot(coords_psi[:, 0], coords_psi[:, 1], coords_psi[:, 2],         'o-', color='tab:red', lw=2, ms=4, label='psi_2 +30 deg')ax2.scatter(*coords_ref[4], color='orange', s=80, marker='s', label='CA2 (axis start)')ax2.scatter(*coords_ref[5], color='gold', s=80, marker='^', label='C2 (axis end)')ax2.set_xlabel('X'); ax2.set_ylabel('Y'); ax2.set_zlabel('Z')ax2.set_title('3D: reference vs psi_2 rotation')ax2.legend(fontsize=7)plt.tight_layout(); plt.show()

### Verify: only the target torsion changedWe extract all backbone dihedrals before and after to confirm only $$\psi_2$$ changed.

In [ ]:
ic_ref = extract_backbone_internal(coords_ref)ic_psi = extract_backbone_internal(coords_psi)print('Torsion changes (before -> after):')for (i, j, k, l, d_old), (_, _, _, _, d_new) in zip(ic_ref.dihedrals, ic_psi.dihedrals):    delta = np.rad2deg(d_new - d_old)    while delta > 180: delta -= 360    while delta < -180: delta += 360    label = f'{names[i]}-{names[j]}-{names[k]}-{names[l]}'    is_target = (i, j, k, l) == psi2_atoms    marker = ' <-- psi_2 TARGET' if is_target else ''    if abs(delta) > 0.5:        print(f'  {label:30s}  {np.rad2deg(d_old):7.1f} -> {np.rad2deg(d_new):7.1f}  (delta = {delta:+6.1f}){marker}')    else:        print(f'  {label:30s}  {np.rad2deg(d_old):7.1f} -> {np.rad2deg(d_new):7.1f}  (delta = {delta:+6.1f})')print(f'\nOK: Only psi_2 changed; all other torsions preserved.')

## 5. Example 2: Rotate $$\phi_3$$ by $$+30^\circ$$$$\phi_3 = C_2 - N_3 - CA_3 - C_3$$ involves atoms 5,6,7,8 (0-based).- **Rotation axis**: $$N_3 \rightarrow CA_3$$ (atoms 6->7)- **Downstream slice**: starts at $$C_3$$ (index 8) -- a shorter block.The N-terminal half should be frozen; atoms 8-14 should move.

In [ ]:
phi3_atoms = (5, 6, 7, 8)    # C2, N3, CA3, C3  (0-based)phi3_slice = 8                # C3 is the first atom that movescoords_phi = apply_torsion_change(    coords_ref, *phi3_atoms[:3], phi3_slice, delta_deg=+30.0)disp2 = np.linalg.norm(coords_phi - coords_ref, axis=1)print(f'Rotating phi_3 by +30 deg\n')print(f'{"atom":>6s}  {"index":>5s}  {"displacement (A)":>18s}  {"moved?":>8s}')print('  ' + '-' * 42)for i, (name, d) in enumerate(zip(names, disp2)):    marker = '  <<<' if i >= phi3_slice else ''    print(f'  {name:>6s}  {i:>5d}  {d:18.4f}{marker}')print(f'\nOK: Atoms 0-7 (N1...CA3) are stationary -- the N-terminal half is frozen.')print(f'OK: Atoms 8-14 (C3...C5) all moved.')print(f'  Max displacement at terminal atom C5: {disp2[-1]:.2f} A')

In [ ]:
# Side-by-side comparison.fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))for ax, d, sl, title in [    (ax1, disp, psi2_slice, 'psi_2 +30 deg (9 atoms move)'),    (ax2, disp2, phi3_slice, 'phi_3 +30 deg (7 atoms move)'),]:    colors = ['0.5' if i < sl else 'tab:red' for i in range(len(names))]    ax.bar(range(len(names)), d, color=colors, edgecolor='white')    ax.axvline(sl - 0.5, color='black', ls='--', lw=1.5)    ax.set_xlabel('atom index'); ax.set_ylabel('displacement (A)')    ax.set_title(title)    ax.set_xticks(range(len(names)))    ax.set_xticklabels(names, rotation=45, fontsize=6)plt.suptitle('Downstream slice comparison', fontsize=13)plt.tight_layout(); plt.show()print(f'\npsi_2 downstream: {len(names) - psi2_slice} atoms moved')print(f'phi_3 downstream: {len(names) - phi3_slice} atoms moved')print(f'\nphi_3 has a SHORTER downstream chain because it sits further down the backbone.')

## 6. Internal $$\leftrightarrow$$ Cartesian: the full round-tripNow let's use the actual conversion pipeline from `core/convert.py`:1. **Cartesian $$\rightarrow$$ Internal**: `cartesian_to_internal` extracts a Z-matrix from Cartesian coordinates.2. **Internal $$\rightarrow$$ Cartesian**: `internal_to_cartesian` rebuilds Cartesian from the Z-matrix using NeRF.The round-trip should be **exact** (RMSD = 0). Let's verify and inspect the Z-matrix.

In [ ]:
# Step 1: Extract the Z-matrix from the reference coordinates.atoms = [(name, *xyz) for name, xyz in zip(names, coords_ref)]peptide_bonds = [(i, i + 1) for i in range(len(atoms) - 1)]zm_entries = cartesian_to_internal(atoms, bonds=peptide_bonds)print("Z-matrix extracted from Cartesian coordinates:")print(f"{'idx':>5s} {'sym':>6s}  {'bond_to':>8s} {'r (A)':>8s}  "      f"{'ang_with':>9s} {'angle':>8s}  {'dih_with':>9s} {'dihedral':>9s}")print('-' * 85)for e in zm_entries:    b = f"{e.bond_to:>3d}  {e.bond_length:6.3f}" if e.bond_to else "  -     -"    a = f"{e.angle_with:>3d}  {np.rad2deg(e.angle):6.1f}" if e.angle_with else "  -     -"    d = f"{e.dihedral_with:>3d}  {np.rad2deg(e.dihedral):6.1f}" if e.dihedral_with else "  -     -"    print(f"  {e.index:3d}  {e.symbol:>6s}  {b}  {a}  {d}")print(f"\nNote: the first 3 atoms have None for reference atoms they cannot define.")print(f"Atom 1 is at origin; atom 2 along +z; atom 3 in the xz-plane.")print(f"Atoms 4+ each have 3 reference atoms: bond_to, angle_with, dihedral_with.")

In [ ]:
# Step 2: Rebuild Cartesian from the Z-matrix and verify round-trip.coords_rt = internal_to_cartesian(zm_entries)rmsd = float(np.sqrt(np.mean(np.sum((coords_ref - coords_rt) ** 2, axis=1))))print(f'Round-trip RMSD (Z-matrix -> Cartesian): {rmsd:.4e} A')if rmsd < 1e-6:    print('OK: exact -- the conversion is lossless.')else:    print(f'Small difference ({rmsd:.2e}) due to floating point.')

## 7. The IUPAC $$\leftrightarrow$$ NeRF sign conventionThis is the most subtle point of the conversion:- **IUPAC** (used by PDB, the `dihedral()` function): looking down B->C, positive dihedral rotates D **clockwise** relative to A.- **NeRF** (used by `place_atom`): the local frame normal points the **opposite** way, so the sign is **negated**.Look at the Z-matrix dihedral values above -- notice they are **negative** of what you'd expect from IUPAC. This is deliberate: `cartesian_to_internal` negates the measured dihedral (`dihedral=-dih` on line 264 of `convert.py`) so that `internal_to_cartesian` can feed it straight into `place_atom` and reconstruct the exact same coordinates.Let's verify: extract all dihedrals in IUPAC convention and compare.

In [ ]:
# Extract full internal coordinates in IUPAC convention.ic = extract_backbone_internal(coords_ref)print("Full internal coordinates (IUPAC convention):")print(f"\n  --- Bond lengths ({len(ic.bonds)}) ---")for i, j, d in ic.bonds[:5]:  # first 5 bonds    print(f"    {names[i]:>4s}-{names[j]:<4s}  {d:.3f} A")print("    ...")print(f"\n  --- Bond angles ({len(ic.angles)}) ---")for i, j, k, a in ic.angles[:5]:  # first 5 angles    print(f"    {names[i]:>4s}-{names[j]:>4s}-{names[k]:<4s}  {np.rad2deg(a):6.1f} deg")print("    ...")print(f"\n  --- Dihedral angles ({len(ic.dihedrals)}) ---")for i, j, k, l, d in ic.dihedrals:    label = f"{names[i]}-{names[j]}-{names[k]}-{names[l]}"    # Identify phi/psi/omega.    if j % 3 == 0 and j + 2 == l:        kind = f"phi_{j//3 + 1}"    elif i % 3 == 0 and i + 3 == l:        kind = f"psi_{i//3 + 1}"    else:        kind = "omega"    print(f"    {label:30s}  {np.rad2deg(d):7.1f} deg   ({kind})")# Verify phi/psi match the input.print(f"\nInput torsions:  phi = {phi_helix},  psi = {psi_helix}")print("Note: extracted phi/psi magnitudes should match the input.")print("The sign may differ because of the IUPAC/NeRF convention -- see the tutorial.")

## 8. Method 2: Change a torsion in the Z-matrix and rebuildInstead of Rodrigues rotation, we can change the `dihedral` field of the relevant Z-matrix entry and call `internal_to_cartesian`. All atoms placed after the changed entry get new positions because their reference frames have been rotated.Let's verify this produces the same result as Rodrigues rotation for $$\psi_2$$.

In [ ]:
# Change the dihedral of N3 (atom index 6, Z-matrix entry index 6).# The dihedral for atom 7 (N3) encodes psi_2 = N2-CA2-C2-N3.delta_rad = np.deg2rad(+30.0)zm_entries[6].dihedral -= delta_rad  # negate because NeRF convention is opposite to IUPAC# Rebuild from the modified Z-matrix.coords_zm = internal_to_cartesian(zm_entries)# Compare with the Rodrigues result.rmsd_methods = float(np.sqrt(np.mean(np.sum((coords_psi - coords_zm) ** 2, axis=1))))print(f'RMSD between Rodrigues and Z-matrix rebuild: {rmsd_methods:.2e} A')if rmsd_methods < 1e-6:    print('OK: The two methods are mathematically identical.')else:    print(f'Small difference ({rmsd_methods:.2e}) due to floating point.')

## 9. The conversion pipeline visualized```Cartesian coords       |       | cartesian_to_internal(atoms, bonds)       |   - measure bond lengths, angles, dihedrals       |   - pick reference atoms (bonded > nearest)       |   - NEGATE dihedral: IUPAC -> NeRF convention       v   Z-matrix (list of ZMatrixEntry)       |       | internal_to_cartesian(entries)       |   - atom 1: origin       |   - atom 2: along +z       |   - atom 3: in xz-plane       |   - atoms 4+: place_atom(a, b, c, bond, angle, torsion)  [NeRF]       vCartesian coords'   (RMSD = 0 -- lossless round-trip)```The entire pipeline is **lossless**: extracting and rebuilding gives back exactly the same coordinates. This is what makes it useful as a coordinate representation for molecular simulation.

## 10. The same principle applies to side-chain $$\chi$$ torsionsSide chains branch off the backbone as trees. Each $$\chi$$ angle is a revolute joint:```Lysine:  CA -- CB -- CG -- CD -- CE -- NZ  chi_1 = N-CA-CB-CG    rotates CG, CD, CE, NZ  (4 atoms)  chi_2 = CA-CB-CG-CD   rotates CD, CE, NZ      (3 atoms)  chi_3 = CB-CG-CD-CE   rotates CE, NZ          (2 atoms)  chi_4 = CG-CD-CE-NZ   rotates NZ only         (1 atom)```Let's illustrate with a simple 5-atom linear chain (analogous to a minimal side chain).

In [ ]:
# Build a simple linear chain: A-B-C-D-E using place_atom.def build_chain(bond, angle, torsion_deg):    # Build a 5-atom linear chain A-B-C-D-E.    coords = np.zeros((5, 3))    ang = np.deg2rad(angle)    tau = np.deg2rad(torsion_deg)    coords[0] = [0, 0, 0]                                          # A at origin    coords[1] = [0, 0, bond]                                       # B along +z    coords[2] = [bond * np.sin(ang), 0, bond - bond * np.cos(ang)] # C in xz-plane    coords[3] = place_atom(coords[0], coords[1], coords[2], bond, ang, tau)  # D via NeRF    coords[4] = place_atom(coords[1], coords[2], coords[3], bond, ang, 0.0)   # E    return coordschain_ref = build_chain(bond=1.5, angle=110.0, torsion_deg=60.0)chain_names = ['A', 'B', 'C', 'D', 'E']print('Reference chain: A-B-C-D-E  (torsion A-B-C-D = 60 deg)')for i, (name, row) in enumerate(zip(chain_names, chain_ref)):    print(f'  [{i}] {name}: ({row[0]:6.3f}, {row[1]:6.3f}, {row[2]:6.3f})')# Change A-B-C-D by +120 deg -> D and E move.chain_new = apply_torsion_change(chain_ref, 0, 1, 2, slice_start=3, delta_deg=+120.0)print(f'\nAfter rotating A-B-C-D by +120 deg (sl_start=3):')for i, (name, row) in enumerate(zip(chain_names, chain_new)):    d = np.linalg.norm(chain_new[i] - chain_ref[i])    marker = '  <<<' if i >= 3 else ''    print(f'  [{i}] {name}: ({row[0]:6.3f}, {row[1]:6.3f}, {row[2]:6.3f})  displ={d:.3f} A{marker}')print(f'\nOK: Atoms A, B, C are stationary (indices < 3).')print(f'OK: Atoms D, E both moved (indices >= 3).')print('\nThis is EXACTLY what RDKit does internally when you call SetDihedralDeg.')

## ExercisesFill in the `...` and run. Compare your results with a neighbour!**E1. Cumulative effect of two torsion changes.** Change $$\psi_2$$ by $$+30^\circ$$ and then $$\phi_3$$ by $$-30^\circ$$ on the *same* copy. Does the final displacement of atom $$C_5$$ equal the sum of the two individual displacements? Why or why not?

In [ ]:
# Start from the reference.coords_double = coords_ref.copy()# Step 1: rotate psi_2 by +30 deg# TODO: apply torsion change for psi_2# coords_double = apply_torsion_change(coords_double, ...)# Step 2: on the already-rotated structure, rotate phi_3 by -30 deg# TODO: apply torsion change for phi_3# Compare displacement of C5 (last atom, index 14).# disp_double = np.linalg.norm(coords_double[-1] - coords_ref[-1])# sum_single = disp[-1] + disp2[-1]# print(f'Sum of individual displacements: {sum_single:.2f} A')# print(f'Displacement after both rotations: {disp_double:.2f} A')# print(f'Are they equal? {abs(disp_double - sum_single) < 0.1}')print('TODO: uncomment and fill in the code above.')

**E2. Displacement as a function of $$\Delta\theta$$.** Rotate $$\psi_2$$ by angles from $$-180^\circ$$ to $$+180^\circ$$ in $$30^\circ$$ steps. Record the displacement of $$C_5$$ (index 14). Plot displacement vs $$\Delta\theta$$. At what angle is the displacement maximum?

In [ ]:
angles = np.arange(-180, 181, 30)displacements = []for delta in angles:    # TODO: apply_torsion_change for psi_2 with delta_deg = delta    # coords_new = apply_torsion_change(coords_ref, *psi2_atoms[:3], psi2_slice, delta_deg=delta)    # d = np.linalg.norm(coords_new[14] - coords_ref[14])    # displacements.append(d)    pass# TODO: plot displacements vs angles# plt.figure(figsize=(7, 4))# plt.plot(angles, displacements, 'o-', lw=1.5)# plt.xlabel('delta_theta (deg)'); plt.ylabel('displacement of C5 (A)')# plt.title('Terminal atom displacement vs torsion change')# plt.grid(alpha=0.3); plt.tight_layout(); plt.show()print('TODO: uncomment and fill in the code above.')

**E3. Different starting conformations.** Build both an $$\alpha$$-helix ($$\phi=-57^\circ, \psi=-47^\circ$$) and an extended strand ($$\phi=-135^\circ, \psi=+135^\circ$$). Apply $$\psi_2 +30^\circ$$ to both and compare the displacements. Why are they different?

In [ ]:
# Build an extended strand.phi_ext = [-135.0] * Lpsi_ext = [135.0] * Lcoords_ext, _ = build_peptide_from_internal(seq, phi_deg=phi_ext, psi_deg=psi_ext)# TODO: apply psi_2 +30 deg to the extended strand# TODO: compute displacement of C5 for both conformations# print(f'Helix C5 displacement:   {disp_helix:.2f} A')# print(f'Extended C5 displacement: {disp_extended:.2f} A')print('TODO: uncomment and fill in the code above.')

**E4. Z-matrix round-trip for phi_3.** Use the Z-matrix approach (Section 8) to change $$\phi_3$$ by $$+30^\circ$$ and verify RMSD = 0 against the Rodrigues result from Section 5. Which Z-matrix entry corresponds to $$\phi_3$$?

In [ ]:
# Hint: phi_3 = C2-N3-CA3-C3. The Z-matrix entry for C3 (atom index 8, 1-based index 9)# has dihedral_with pointing to C2, angle_with to N3, bond_to to CA3.# TODO: find the right Z-matrix entry and change its dihedral# zm_entries[...].dihedral -= np.deg2rad(+30.0)# coords_zm2 = internal_to_cartesian(zm_entries)# TODO: compute RMSD with coords_phi from Section 5# rmsd = ...# print(f'RMSD between Rodrigues and Z-matrix rebuild: {rmsd:.2e} A')print('TODO: uncomment and fill in the code above.')

---### Where to go next- **Loop closure**: run `kinematics_loop/examples/example_loop.py` and trace a CCD sweep -- which torsion is rotated, and which atoms move?- **Side-chain packing**: look at `rotamer/core/peptide.py:set_chi` -- how does RDKit implement the same Rodrigues rotation?- **Arbitrary Z-matrix**: try `cartesian_to_internal` on a non-peptide molecule (e.g. from an SDF), change one dihedral, and rebuild with `internal_to_cartesian`.- **Kinematic tree**: implement a general `downstream_subtree(torsion_atoms, bond_graph)` function for branched molecules.See the closing section of `TorsionPropagation_tutorial.md` for more.